# Task 3

Due to the simplicity of KNN for Classification, let's focus on using a Pipeline and a GridSearchCV tool, since these skills can be generalized for any model.


## The Sonar Data

### Detecting a Rock or a Mine

Sonar (sound navigation ranging) is a technique that uses sound propagation (usually underwater, as in submarine navigation) to navigate, communicate with or detect objects on or under the surface of the water, such as other vessels.



The data set contains the response metrics for 60 separate sonar frequencies sent out against a known mine field (and known rocks). These frequencies are then labeled with the known object they were beaming the sound at (either a rock or a mine).



Our main goal is to create a machine learning model capable of detecting the difference between a rock or a mine based on the response of the 60 separate sonar frequencies.


Data Source: https://archive.ics.uci.edu/ml/datasets/Connectionist+Bench+(Sonar,+Mines+vs.+Rocks)

### Complete the Tasks in bold

**TASK: Run the cells below to load the data.**

In [24]:
import numpy as np
import pandas as pd

In [25]:
df = pd.read_csv('sonar.all-data.csv')

## Train | Test Split

Our approach here will be one of using Cross Validation on 90% of the dataset, and then judging our results on a final test set of 10% to evaluate our model.

**TASK: Split the data into features and labels, and then split into a training set and test set, with 90% for Cross-Validation training, and 10% for a final test set.**

*Note: The solution uses a random_state=42*

In [26]:
from sklearn.model_selection import train_test_split

# Load the dataset


# Split into features (X) and labels (y)
X = df.iloc[:, :-1]   # All columns except the last
y = df.iloc[:, -1]    # Last column is the label

# Split into 90% training (for cross-validation) and 10% final test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,      # 10% final test set
    random_state=42,    # reproducibility
    stratify=y          # maintain class balance
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Training set shape: (187, 60)
Test set shape: (21, 60)


**TASK: Create a Pipeline that contains both a StandardScaler and a KNN model**

In [27]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Create pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),   # Step 1: Feature scaling
    ('knn', KNeighborsClassifier()) # Step 2: KNN model
])

print(pipeline)

Pipeline(steps=[('scaler', StandardScaler()), ('knn', KNeighborsClassifier())])


**TASK: Perform a grid-search with the pipeline to test various values of k and report back the best performing parameters.**

In [30]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Create pipeline
# Define parameter grid for k
param_grid = {
    'knn__n_neighbors': range(1, 100),
}
# Set up GridSearch with 5-fold cross-validation
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='accuracy'
)

# Fit on the 90% training data
grid_search.fit(X_train, y_train)

# Best parameters and score
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

Best Parameters: {'knn__n_neighbors': 1}
Best Cross-Validation Accuracy: 0.8398293029871977


### Final Model Evaluation

**TASK: Using the grid classifier object from the previous step, get a final performance classification report and confusion matrix.**

In [31]:
from sklearn.metrics import classification_report, confusion_matrix

# Use the best model found by GridSearch
best_model = grid_search.best_estimator_

# Make predictions on the final test set
y_pred = best_model.predict(X_test)

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           M       0.77      0.91      0.83        11
           R       0.88      0.70      0.78        10

    accuracy                           0.81        21
   macro avg       0.82      0.80      0.81        21
weighted avg       0.82      0.81      0.81        21

Confusion Matrix:
[[10  1]
 [ 3  7]]


In [ ]:
#Confusion matrix shows:
#10 M correctly identified, 1 M misclassified as R
#7 R correctly identified, 3 R misclassified as M
#Total errors: 4 misclassifications

In [ ]:
#Model excels at identifying Class M (91% recall) but misses 30% of Class R cases. 
#Precision is strong for R (88%) but moderate for M (77%).
#Overall, reliable for M detection, less so for R.